In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.impute import SimpleImputer
%matplotlib inline

In [5]:
df = pd.read_csv("best_data.csv")

In [10]:
PAPER_FEATURES = ['Latitude_Degrees', 'Longitude_Degrees', 'Depth_m', 'ClimSST', 'SSTA']
TARGET = 'Percent_Bleaching'
df = df.dropna(subset=['Percent_Bleaching'])

In [14]:
X = df[PAPER_FEATURES]
Y = df[TARGET]

In [18]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

In [ ]:
imputer = SimpleImputer(strategy='median')
X_train_imp = imputer.fit_transform(X_train_raw)
X_test_imp = imputer.transform(X_test_raw)

# scaler = StandardScaler()
# X_train_ss = scaler.fit_transform(X_train_imp)
# X_test_ss = scaler.transform(X_test_imp)


In [24]:
import optuna
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor, VotingRegressor, HistGradientBoostingRegressor
import xgboost as xgb
from sklearn.metrics import mean_absolute_error

# OpTuna

In [21]:
def et_objective(trial):
    model = ExtraTreesRegressor(
        n_estimators     = trial.suggest_int("n_estimators", 50, 300),
        max_depth        = trial.suggest_int("max_depth", 5, 40),
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 5),
        max_features     = trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        n_jobs=-1, random_state=42
    )
    pipeline = Pipeline([
        ('model',   model)
    ])
    scores = cross_val_score(pipeline, X_train_imp, y_train, cv=4, scoring='r2', n_jobs=-1)
    return scores.mean()

et_study = optuna.create_study(direction="maximize")
et_study.optimize(et_objective, n_trials=50)
print("========== ET BEST ==========")
print(et_study.best_params)
print(f"Best R²: {et_study.best_value:.4f}")

[I 2026-05-06 16:29:59,299] A new study created in memory with name: no-name-3658633b-977c-463c-b36e-873cf0ad88c8
[I 2026-05-06 16:30:01,197] Trial 0 finished with value: 0.2937665074128697 and parameters: {'n_estimators': 210, 'max_depth': 12, 'min_samples_leaf': 3, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.2937665074128697.
[I 2026-05-06 16:30:02,681] Trial 1 finished with value: 0.1433386774552072 and parameters: {'n_estimators': 124, 'max_depth': 7, 'min_samples_leaf': 3, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.2937665074128697.
[I 2026-05-06 16:30:04,280] Trial 2 finished with value: 0.1714800659451313 and parameters: {'n_estimators': 190, 'max_depth': 8, 'min_samples_leaf': 3, 'max_features': 'log2'}. Best is trial 0 with value: 0.2937665074128697.
[I 2026-05-06 16:30:05,104] Trial 3 finished with value: 0.39513644400523185 and parameters: {'n_estimators': 290, 'max_depth': 39, 'min_samples_leaf': 5, 'max_features': 'sqrt'}. Best is trial 3 with value:

========== ET BEST ==========
{'n_estimators': 277, 'max_depth': 26, 'min_samples_leaf': 1, 'max_features': 'sqrt'}
Best R²: 0.5956


In [22]:
def rf_objective(trial):
    model = RandomForestRegressor(
        n_estimators      = trial.suggest_int("n_estimators", 50, 300),
        max_depth         = trial.suggest_int("max_depth", 5, 40),
        min_samples_split = trial.suggest_int("min_samples_split", 2, 10),
        min_samples_leaf  = trial.suggest_int("min_samples_leaf", 1, 5),
        max_features      = trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        n_jobs=-1, random_state=42
    )
    pipeline = Pipeline([
        ('model',   model)
    ])
    scores = cross_val_score(pipeline, X_train_imp, y_train, cv=4, scoring='r2', n_jobs=-1)
    return scores.mean()

rf_study = optuna.create_study(direction="maximize")
rf_study.optimize(rf_objective, n_trials=50)
print("========== RF BEST ==========")
print(rf_study.best_params)
print(f"Best R²: {rf_study.best_value:.4f}")

[I 2026-05-06 16:30:47,896] A new study created in memory with name: no-name-5e699d9f-d98a-4bde-8dec-c7330e3fa25d
[I 2026-05-06 16:30:49,435] Trial 0 finished with value: 0.4962275847151242 and parameters: {'n_estimators': 254, 'max_depth': 11, 'min_samples_split': 4, 'min_samples_leaf': 2, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.4962275847151242.
[I 2026-05-06 16:30:50,191] Trial 1 finished with value: 0.5028561557324032 and parameters: {'n_estimators': 113, 'max_depth': 13, 'min_samples_split': 6, 'min_samples_leaf': 5, 'max_features': 'log2'}. Best is trial 1 with value: 0.5028561557324032.
[I 2026-05-06 16:30:50,862] Trial 2 finished with value: 0.49830047948955913 and parameters: {'n_estimators': 106, 'max_depth': 12, 'min_samples_split': 8, 'min_samples_leaf': 4, 'max_features': 'sqrt'}. Best is trial 1 with value: 0.5028561557324032.
[I 2026-05-06 16:30:52,517] Trial 3 finished with value: 0.5340831726074434 and parameters: {'n_estimators': 225, 'max_depth': 38, '

========== RF BEST ==========
{'n_estimators': 275, 'max_depth': 33, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'log2'}
Best R²: 0.5995


In [25]:
def hgb_objective(trial):
    model = HistGradientBoostingRegressor(
        max_iter          = trial.suggest_int  ("max_iter",          50, 400),
        max_depth         = trial.suggest_int  ("max_depth",          3,  15),
        learning_rate     = trial.suggest_float("learning_rate",  0.005, 0.1),
        min_samples_leaf  = trial.suggest_int  ("min_samples_leaf",   1,  50),
        l2_regularization = trial.suggest_float("l2_regularization", 0.0, 1.0),
        random_state=42
    )
    pipeline = Pipeline([
        ('model',   model)
    ])
    scores = cross_val_score(pipeline, X_train_imp, y_train, cv=4, scoring='r2', n_jobs=-1)
    return scores.mean()

hgb_study = optuna.create_study(direction="maximize")
hgb_study.optimize(hgb_objective, n_trials=50)
print("========== HGB BEST ==========")
print(hgb_study.best_params)
print(f"Best R²: {hgb_study.best_value:.4f}")

[I 2026-05-06 16:32:35,409] A new study created in memory with name: no-name-bb963f84-ad2f-4ae9-858a-11995c0bbcd3
[I 2026-05-06 16:32:36,091] Trial 0 finished with value: 0.4851160308207723 and parameters: {'max_iter': 357, 'max_depth': 13, 'learning_rate': 0.04143907173092881, 'min_samples_leaf': 16, 'l2_regularization': 0.6166561129891761}. Best is trial 0 with value: 0.4851160308207723.
[I 2026-05-06 16:32:36,702] Trial 1 finished with value: 0.28578532017827035 and parameters: {'max_iter': 295, 'max_depth': 5, 'learning_rate': 0.005433335179995715, 'min_samples_leaf': 8, 'l2_regularization': 0.25905360647836095}. Best is trial 0 with value: 0.4851160308207723.
[I 2026-05-06 16:32:36,919] Trial 2 finished with value: 0.2784225323704982 and parameters: {'max_iter': 87, 'max_depth': 14, 'learning_rate': 0.012824875060929317, 'min_samples_leaf': 49, 'l2_regularization': 0.5732064373464307}. Best is trial 0 with value: 0.4851160308207723.
[I 2026-05-06 16:32:37,826] Trial 3 finished wit

========== HGB BEST ==========
{'max_iter': 381, 'max_depth': 11, 'learning_rate': 0.08385980706077449, 'min_samples_leaf': 4, 'l2_regularization': 0.5182337453144167}
Best R²: 0.5302


# Start

In [26]:
et_model = ExtraTreesRegressor(
    n_estimators=277,
    max_depth=26,
    min_samples_leaf=1,
    max_features='sqrt',
    n_jobs=-1,
    random_state=42
)
# {'n_estimators': 277, 'max_depth': 26, 'min_samples_leaf': 1, 'max_features': 'sqrt'}

rf_model = RandomForestRegressor(
    n_estimators=275,
    max_depth=33,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features='log2',
    n_jobs=-1,
    random_state=42
)
# {'n_estimators': 275, 'max_depth': 33, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'log2'}

hgb_model = HistGradientBoostingRegressor(
    max_iter=381,
    max_depth=11,
    learning_rate=0.08386,
    min_samples_leaf=4,
    l2_regularization=0.518,
    random_state=42
)
# {'max_iter': 381, 'max_depth': 11, 'learning_rate': 0.08385980706077449, 'min_samples_leaf': 4, 'l2_regularization': 0.5182337453144167}


In [27]:
voting_model = VotingRegressor([
    ('et',  et_model),
    ('rf',  rf_model),
    ('hgb', hgb_model)
])

# Train all 4
et_model.fit(X_train_imp, y_train)
rf_model.fit(X_train_imp, y_train)
hgb_model.fit(X_train_imp, y_train)
voting_model.fit(X_train_imp, y_train)

# Evaluate
models = {
    'ExtraTrees':        et_model,
    'RandomForest':      rf_model,
    'HistGradienBoosting':           hgb_model,
    'Voting (Ensemble)': voting_model
}

print("=" * 65)
print("BASELINE RESULTS (single seed = 42)")
print("=" * 65)
print(f"{'Model':<22}{'R²':>10}{'RMSE':>12}{'MAE':>10}{'Acc%':>10}")
print("-" * 65)
for name, model in models.items():
    y_pred = model.predict(X_test_imp)
    r2   = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae  = mean_absolute_error(y_test, y_pred)
    print(f"{name:<22}{r2:>10.4f}{rmse:>12.4f}{mae:>10.4f}{100-mae:>10.2f}")
print("-" * 65)
print(f"{'Paper baseline:':<22}{0.6395:>10.4f}{12.9268:>12.4f}{'—':>10}{93.88:>10.2f}")
print("=" * 65)

BASELINE RESULTS (single seed = 42)
Model                         R²        RMSE       MAE      Acc%
-----------------------------------------------------------------
ExtraTrees                0.6632     12.2458    5.5985     94.40
RandomForest              0.6515     12.4575    6.2161     93.78
HistGradienBoosting       0.5393     14.3219    8.1476     91.85
Voting (Ensemble)         0.6445     12.5822    6.5536     93.45
-----------------------------------------------------------------
Paper baseline:           0.6395     12.9268         —     93.88


# K-Fold

In [28]:
from sklearn.model_selection import KFold, cross_validate
from sklearn.pipeline import Pipeline

cv = KFold(n_splits=4, shuffle=True, random_state=42)

def make_pipeline(model):
    return Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model',   model)
    ])

cv_models = {
    'ExtraTrees':        make_pipeline(et_model),
    'RandomForest':      make_pipeline(rf_model),
    'HistGradientBoost': make_pipeline(hgb_model),
    'Voting (Ensemble)': make_pipeline(voting_model)
}

print("=" * 70)
print("K-Fold Cross Validation Summary (k=4)")
print("=" * 70)
print(f"{'Model':<22}{'Mean R²':>14}{'Mean RMSE':>14}{'Mean MAE':>14}")
print("-" * 70)

for name, model in cv_models.items():
    cv_results = cross_validate(
        model, X, Y, cv=cv,
        scoring={'r2': 'r2',
                 'neg_mse': 'neg_mean_squared_error',
                 'neg_mae': 'neg_mean_absolute_error'},
        n_jobs=-1
    )
    r2_mean   = cv_results['test_r2'].mean()
    rmse_mean = np.sqrt(-cv_results['test_neg_mse']).mean()
    mae_mean  = (-cv_results['test_neg_mae']).mean()
    print(f"{name:<22}{r2_mean:>14.4f}{rmse_mean:>14.4f}{mae_mean:>14.4f}")

print("=" * 70)

K-Fold Cross Validation Summary (k=4)
Model                        Mean R²     Mean RMSE      Mean MAE
----------------------------------------------------------------------
ExtraTrees                    0.6406       12.9067        5.9917
RandomForest                  0.6328       13.0461        6.5577
HistGradientBoost             0.5354       14.6709        8.3773
Voting (Ensemble)             0.6282       13.1256        6.8641
